# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Record Sets: {[rset.id for rset in metadata.record_sets]}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

In [ ]:
# List all RecordSet @ids, names, and their fields
for record_set in metadata.record_sets:
    print(f"RecordSet @id: {record_set.id}, name: {record_set.name}")
    print("  Fields:")
    for field in record_set.fields:
        col_ids = [col.id for col in getattr(field, 'columns', [])] if hasattr(field, 'columns') else []
        print(f"   - Field @id: {field.id}, name: {field.name}, columns: {col_ids}")
    print("  Columns:")
    for column in record_set.columns:
        print(f"   - Column @id: {column.id}, name: {getattr(column, 'name', 'N/A')}")
    print("\n")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Example: Extract all available record sets dynamically
record_set_ids = [rset.id for rset in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for {record_set_id} with columns: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    else:
        print(f"No records found for RecordSet @id: {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# -- Parameters: Please adapt these based on the printed record set and field IDs above --
# For demonstration, we choose the first record set and try to pick a numeric field/column

if len(dataframes) > 0:
    # Select a record set (first available)
    main_record_set_id = record_set_ids[0]
    print(f"Using RecordSet @id: {main_record_set_id}")

    df = dataframes[main_record_set_id]
    print(f"Columns: {df.columns.tolist()}")

    # Attempt to select a numeric field automatically (e.g., any column with float/int dtype or containing 'abundance', 'MW', etc.)
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_candidates:
        # Try fallback by known names
        for key in ['Abundance', 'MW', 'MolecularWeight', 'Peptides', 'coverage']:
            for col in df.columns:
                if key.lower() in col.lower():
                    numeric_candidates.append(col)
        numeric_candidates = list(set(numeric_candidates))

    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Chosen numeric field for EDA: {numeric_field}")
        # Filter on numeric_field > threshold (for demo, choose threshold as mean value)
        try:
            threshold = df[numeric_field].mean()
            print(f"Filtering {numeric_field} > {threshold}")
            filtered_df = df[df[numeric_field] > threshold].copy()
            print(f"Filtered records with {numeric_field} > {threshold}:")
            display(filtered_df.head())

            # Normalize this field
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Group by a likely categorical field (e.g., columns with string/object dtype)
            group_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
            group_field = group_candidates[0] if group_candidates else None
            if group_field:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
                print(f"Grouped data by {group_field} (mean {numeric_field}):")
                display(grouped_df.head())
        except Exception as e:
            print(f"Error in EDA: {e}")
    else:
        print("No numeric field found for analysis.")
else:
    print("No dataframes were loaded. Please check the Dataset or its record sets.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Example: Histogram of the selected numeric field and boxplot grouped by a categorical field
import matplotlib.pyplot as plt
import seaborn as sns
if 'filtered_df' in locals() and not filtered_df.empty:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(12, 6))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No filtered DataFrame available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we loaded the FAIR^2 Croissant dataset describing mass spectrometry-based human protein data from extracellular vesicles. We examined the structure via record set, field, and column `@id`s. Using `mlcroissant`, we loaded tabular data, automated basic EDA, filtered and normalized quantitative variables, and visualized the key distributions. These steps serve as a foundation for more advanced biomarker discovery, statistical profiling, or machine learning workflows on this rich dataset.*